In [2]:
import pandas as pd 
from pathlib import Path



In [2]:
PROJECT_ROOT = Path.cwd().parent

HMIS_RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Hospital HMIS Dataset for Healthcare Analytics"
    / "hospital_synthetic_shalaka"
    / "hospital_data"
)
print(HMIS_RAW_DATA_DIR)

C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\raw\Hospital HMIS Dataset for Healthcare Analytics\hospital_synthetic_shalaka\hospital_data


In [3]:
file_names = [
    
    # Parent / Master Tables
    "department.csv",
    "patient.csv",
    "employee.csv",
    "disease.csv",
    "insurance_provider.csv",
    "drug_manufacturer.csv",

    # First-Level Dependent Tables
    "doctor.csv",
    "ward.csv",
    "drug.csv",
    "patient_insurance.csv",

    # Operational Resources
    "bed.csv",
    "drug_inventory.csv",

    # Hospital Transactions
    "staff_assignment.csv",
    "admission.csv",

    # Clinical Transactions
    "diagnostic_test.csv",
    "patient_diagnostic.csv",
    "prescription.csv",

    # Financial Transactions
    "billing.csv",
    "billing_detail.csv"
]
dataframes ={} #RAW data
for file_name in file_names:
    file_path = HMIS_RAW_DATA_DIR / file_name
    # Load CSV
    df = pd.read_csv(file_path)
    key_name=file_name[0:-4]
    dataframes[key_name]=df

#cleaned data
cleaned_dataframes = {
    table_name: df.copy(deep=True)
    for table_name, df in dataframes.items()
}

Cleaning department.csv

In [4]:
# =========================================================
# HELPER FUNCTIONS — reused for every table, kept simple on purpose
# =========================================================

def strip_string_columns(df):
    """Remove leading/trailing spaces from every text column.
    Doesn't change values, just removes invisible formatting issues
    that can break joins/grouping later (e.g. 'Male ' != 'Male')."""
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
    return df

def convert_to_datetime(df, date_cols):
    """Convert text date columns to real datetime type.
    errors='coerce' turns any unparseable date into NaT instead of
    crashing, so we can spot bad dates instead of losing the row."""
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

def report_missing(df, table_name):
    """Print % missing per column, only if any exist."""
    pct = (df.isnull().sum() / len(df) * 100).round(2)
    pct = pct[pct > 0]
    if len(pct) > 0:
        print(f"[{table_name}] missing values:\n{pct}\n")
    else:
        print(f"[{table_name}] no missing values")

def report_duplicates(df, table_name):
    """Print count of fully duplicated rows."""
    n = df.duplicated().sum()
    print(f"[{table_name}] duplicate rows: {n}")

def check_foreign_key(child_df, child_col, parent_df, parent_col, child_name, parent_name):
    """Check every value in child_col exists in the parent table.
    We only REPORT orphans here — we never silently drop rows,
    since that would break the joins other tables depend on."""
    orphans = ~child_df[child_col].isin(parent_df[parent_col])
    print(f"[{child_name}.{child_col} -> {parent_name}.{parent_col}] orphan rows: {orphans.sum()}")
    return orphans


# =========================================================
# PARENT / MASTER TABLES
# =========================================================

In [5]:
# --- department ---

df = cleaned_dataframes['department']
df = strip_string_columns(df)
report_missing(df, 'department')
report_duplicates(df, 'department')
cleaned_dataframes['department'] = df


[department] no missing values
[department] duplicate rows: 0


In [6]:
# --- patient ---
df = cleaned_dataframes['patient']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['date_of_birth'])

# contact_number comes in many formats (with/without country code,
# dashes, dots, brackets, extensions like 'x90630').
# We keep the ORIGINAL column untouched, and add a clean digits-only
# version for anything that needs a consistent phone format.
df['contact_number_clean'] = (
    df['contact_number']
    .str.split('x').str[0]          # drop extension part like x90630
    .str.replace(r'\D', '', regex=True)  # keep digits only
    .str[-10:]                      # keep last 10 digits (local number)
)

report_missing(df, 'patient')
report_duplicates(df, 'patient')
cleaned_dataframes['patient'] = df
print(df.head())

[patient] no missing values
[patient] duplicate rows: 0
   patient_id  gender date_of_birth blood_group                city  \
0           1  Female    1987-08-24          O-  East Stephanieberg   
1           2    Male    1960-05-18          A-          Manuelbury   
2           3    Male    1955-04-24          A-   Lake Susanchester   
3           4    Male    2004-06-16          B-   South Leslieburgh   
4           5    Male    1977-07-22          A-        Lopezchester   

        contact_number contact_number_clean  
0      +1-792-342-0981           7923420981  
1         793-725-0800           7937250800  
2  +1-330-617-3749x232           3306173749  
3   426-611-6235x07684           4266116235  
4   (215)330-2831x0821           2153302831  


In [7]:
# --- employee ---
df = cleaned_dataframes['employee']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['date_of_joining'])
report_missing(df, 'employee')
report_duplicates(df, 'employee')
cleaned_dataframes['employee'] = df
print(df.info())

[employee] no missing values
[employee] duplicate rows: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   employee_id      500 non-null    int64         
 1   employee_name    500 non-null    object        
 2   gender           500 non-null    object        
 3   role             500 non-null    object        
 4   employment_type  500 non-null    object        
 5   date_of_joining  500 non-null    datetime64[ns]
 6   department_id    500 non-null    int64         
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 27.5+ KB
None


In [8]:
# --- disease ---
df = cleaned_dataframes['disease']
df = strip_string_columns(df)
report_missing(df, 'disease')
report_duplicates(df, 'disease')
cleaned_dataframes['disease'] = df

[disease] no missing values
[disease] duplicate rows: 0


In [9]:
# --- insurance_provider ---
df = cleaned_dataframes['insurance_provider']
df = strip_string_columns(df)
report_missing(df, 'insurance_provider')
report_duplicates(df, 'insurance_provider')
cleaned_dataframes['insurance_provider'] = df

[insurance_provider] no missing values
[insurance_provider] duplicate rows: 0


In [10]:
# --- drug_manufacturer ---
df = cleaned_dataframes['drug_manufacturer']
df = strip_string_columns(df)
report_missing(df, 'drug_manufacturer')
report_duplicates(df, 'drug_manufacturer')
cleaned_dataframes['drug_manufacturer'] = df

[drug_manufacturer] no missing values
[drug_manufacturer] duplicate rows: 0


# =========================================================
# FIRST-LEVEL DEPENDENT TABLES
# =========================================================

In [11]:
# --- doctor ---
df = cleaned_dataframes['doctor']
df = strip_string_columns(df)
check_foreign_key(df, 'employee_id', cleaned_dataframes['employee'], 'employee_id', 'doctor', 'employee')
report_missing(df, 'doctor')
report_duplicates(df, 'doctor')
cleaned_dataframes['doctor'] = df

[doctor.employee_id -> employee.employee_id] orphan rows: 0
[doctor] no missing values
[doctor] duplicate rows: 0


In [12]:
# --- ward ---
df = cleaned_dataframes['ward']
df = strip_string_columns(df)
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'ward', 'department')
report_missing(df, 'ward')
report_duplicates(df, 'ward')
cleaned_dataframes['ward'] = df

[ward.department_id -> department.department_id] orphan rows: 0
[ward] no missing values
[ward] duplicate rows: 0


In [13]:
# --- drug ---
df = cleaned_dataframes['drug']
df = strip_string_columns(df)
check_foreign_key(df, 'manufacturer_id', cleaned_dataframes['drug_manufacturer'], 'manufacturer_id', 'drug', 'drug_manufacturer')
report_missing(df, 'drug')
report_duplicates(df, 'drug')
cleaned_dataframes['drug'] = df

[drug.manufacturer_id -> drug_manufacturer.manufacturer_id] orphan rows: 0
[drug] no missing values
[drug] duplicate rows: 0


In [14]:
# --- patient_insurance ---
df = cleaned_dataframes['patient_insurance']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['policy_start_date', 'policy_end_date'])
check_foreign_key(df, 'patient_id', cleaned_dataframes['patient'], 'patient_id', 'patient_insurance', 'patient')
check_foreign_key(df, 'insurance_provider_id', cleaned_dataframes['insurance_provider'], 'insurance_provider_id', 'patient_insurance', 'insurance_provider')
report_missing(df, 'patient_insurance')
report_duplicates(df, 'patient_insurance')
cleaned_dataframes['patient_insurance'] = df

[patient_insurance.patient_id -> patient.patient_id] orphan rows: 0
[patient_insurance.insurance_provider_id -> insurance_provider.insurance_provider_id] orphan rows: 0
[patient_insurance] no missing values
[patient_insurance] duplicate rows: 0


# =========================================================
# OPERATIONAL RESOURCES
# =========================================================

In [15]:
# --- bed ---
df = cleaned_dataframes['bed']
df = strip_string_columns(df)
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'bed', 'ward')
report_missing(df, 'bed')
report_duplicates(df, 'bed')
cleaned_dataframes['bed'] = df

[bed.ward_id -> ward.ward_id] orphan rows: 0
[bed] no missing values
[bed] duplicate rows: 0


In [16]:
# --- drug_inventory ---
df = cleaned_dataframes['drug_inventory']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['last_restock_date'])
check_foreign_key(df, 'drug_id', cleaned_dataframes['drug'], 'drug_id', 'drug_inventory', 'drug')
report_missing(df, 'drug_inventory')
report_duplicates(df, 'drug_inventory')
cleaned_dataframes['drug_inventory'] = df

[drug_inventory.drug_id -> drug.drug_id] orphan rows: 0
[drug_inventory] no missing values
[drug_inventory] duplicate rows: 0


# =========================================================
# HOSPITAL TRANSACTIONS
# =========================================================

In [17]:
# --- staff_assignment ---
df = cleaned_dataframes['staff_assignment']
df = strip_string_columns(df)
check_foreign_key(df, 'employee_id', cleaned_dataframes['employee'], 'employee_id', 'staff_assignment', 'employee')
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'staff_assignment', 'ward')
report_missing(df, 'staff_assignment')
report_duplicates(df, 'staff_assignment')
cleaned_dataframes['staff_assignment'] = df

[staff_assignment.employee_id -> employee.employee_id] orphan rows: 0
[staff_assignment.ward_id -> ward.ward_id] orphan rows: 0
[staff_assignment] no missing values
[staff_assignment] duplicate rows: 0


In [18]:
# --- admission (this is the spine table — extra care here) ---
df = cleaned_dataframes['admission']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['admission_date', 'discharge_date'])

# Sanity check: discharge should never be before admission.
# We flag it instead of silently fixing it, since a bad row here
# would need investigation, not a guess.
bad_dates = df[df['discharge_date'] < df['admission_date']]
print(f"[admission] rows with discharge before admission: {len(bad_dates)}")

check_foreign_key(df, 'patient_id', cleaned_dataframes['patient'], 'patient_id', 'admission', 'patient')
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'admission', 'department')
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'admission', 'ward')
check_foreign_key(df, 'bed_id', cleaned_dataframes['bed'], 'bed_id', 'admission', 'bed')
check_foreign_key(df, 'disease_id', cleaned_dataframes['disease'], 'disease_id', 'admission', 'disease')

report_missing(df, 'admission')
report_duplicates(df, 'admission')
cleaned_dataframes['admission'] = df

[admission] rows with discharge before admission: 0
[admission.patient_id -> patient.patient_id] orphan rows: 0
[admission.department_id -> department.department_id] orphan rows: 0
[admission.ward_id -> ward.ward_id] orphan rows: 0
[admission.bed_id -> bed.bed_id] orphan rows: 0
[admission.disease_id -> disease.disease_id] orphan rows: 0
[admission] no missing values
[admission] duplicate rows: 0


# =========================================================
# CLINICAL TRANSACTIONS
# =========================================================

In [19]:
# --- diagnostic_test ---
df = cleaned_dataframes['diagnostic_test']
df = strip_string_columns(df)
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'diagnostic_test', 'department')
report_missing(df, 'diagnostic_test')
report_duplicates(df, 'diagnostic_test')
cleaned_dataframes['diagnostic_test'] = df

[diagnostic_test.department_id -> department.department_id] orphan rows: 0
[diagnostic_test] no missing values
[diagnostic_test] duplicate rows: 0


In [20]:
# --- patient_diagnostic ---
df = cleaned_dataframes['patient_diagnostic']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['test_date'])
check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'patient_diagnostic', 'admission')
check_foreign_key(df, 'test_id', cleaned_dataframes['diagnostic_test'], 'test_id', 'patient_diagnostic', 'diagnostic_test')
check_foreign_key(df, 'doctor_id', cleaned_dataframes['doctor'], 'doctor_id', 'patient_diagnostic', 'doctor')
report_missing(df, 'patient_diagnostic')
report_duplicates(df, 'patient_diagnostic')
cleaned_dataframes['patient_diagnostic'] = df

[patient_diagnostic.admission_id -> admission.admission_id] orphan rows: 0
[patient_diagnostic.test_id -> diagnostic_test.test_id] orphan rows: 0
[patient_diagnostic.doctor_id -> doctor.doctor_id] orphan rows: 0
[patient_diagnostic] no missing values
[patient_diagnostic] duplicate rows: 0


In [21]:
# --- prescription ---
df = cleaned_dataframes['prescription']
df = strip_string_columns(df)
check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'prescription', 'admission')
check_foreign_key(df, 'drug_id', cleaned_dataframes['drug'], 'drug_id', 'prescription', 'drug')
report_missing(df, 'prescription')
report_duplicates(df, 'prescription')
cleaned_dataframes['prescription'] = df

[prescription.admission_id -> admission.admission_id] orphan rows: 0
[prescription.drug_id -> drug.drug_id] orphan rows: 0
[prescription] no missing values
[prescription] duplicate rows: 0


# =========================================================
# FINANCIAL TRANSACTIONS
# =========================================================

In [22]:
# --- billing ---
df = cleaned_dataframes['billing']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['bill_date'])

# Sanity check: insurance_covered + patient_payable should add up
# to roughly total_amount. We just flag mismatches, don't fix them.
mismatch = (
    (df['insurance_covered_amount'] + df['patient_payable_amount'] - df['total_amount']).abs() > 1
)
print(f"[billing] rows where covered+payable != total: {mismatch.sum()}")

check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'billing', 'admission')
report_missing(df, 'billing')
report_duplicates(df, 'billing')
cleaned_dataframes['billing'] = df

[billing] rows where covered+payable != total: 0
[billing.admission_id -> admission.admission_id] orphan rows: 0
[billing] no missing values
[billing] duplicate rows: 0


In [23]:
# --- billing_detail ---
df = cleaned_dataframes['billing_detail']
df = strip_string_columns(df)

# reference_id is NaN for charge types like 'Room' that don't point
# to a specific drug/test — that's expected, not an error.
# We convert it to pandas' nullable integer type (Int64) so it stays
# an ID-like whole number instead of an awkward float (76.0).
df['reference_id'] = df['reference_id'].astype('Int64')

check_foreign_key(df, 'bill_id', cleaned_dataframes['billing'], 'bill_id', 'billing_detail', 'billing')
report_missing(df, 'billing_detail')
report_duplicates(df, 'billing_detail')
cleaned_dataframes['billing_detail'] = df

[billing_detail.bill_id -> billing.bill_id] orphan rows: 0
[billing_detail] missing values:
reference_id    59.97
dtype: float64

[billing_detail] duplicate rows: 0


# =========================================================
# SAVE CLEANED TABLES
# =========================================================

In [24]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "Hospital HMIS Dataset for Healthcare Analytics"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for table_name, df in cleaned_dataframes.items():
    out_path = PROCESSED_DIR / f"{table_name}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved {table_name} -> {out_path}")

Saved department -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\Hospital HMIS Dataset for Healthcare Analytics\department.csv
Saved patient -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\Hospital HMIS Dataset for Healthcare Analytics\patient.csv
Saved employee -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\Hospital HMIS Dataset for Healthcare Analytics\employee.csv
Saved disease -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\Hospital HMIS Dataset for Healthcare Analytics\disease.csv
Saved insurance_provider -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\Hospital HMIS Dataset for Healthcare Analytics\insurance_provider.csv
Saved drug_manufacturer -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\Hospi

NOW WE ARE MAKING FOUR TABLES REQUIRED OF DASHBOARD

In [32]:
# =========================================================
# LOAD CLEANED HMIS TABLES FROM data/processed/
# (dates get saved as text in CSV, so we re-parse them here)
# =========================================================
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" 

cleaned_dataframes = {}
for name in ['department','patient','employee','disease','insurance_provider','drug_manufacturer',
             'doctor','ward','drug','patient_insurance','bed','drug_inventory','staff_assignment',
             'admission','diagnostic_test','patient_diagnostic','prescription','billing','billing_detail']:
    cleaned_dataframes[name] = pd.read_csv(PROCESSED_DIR / "Hospital HMIS Dataset for Healthcare Analytics" / f"{name}.csv")

date_cols_by_table = {
    'patient': ['date_of_birth'],
    'employee': ['date_of_joining'],
    'admission': ['admission_date', 'discharge_date'],
    'patient_insurance': ['policy_start_date', 'policy_end_date'],
    'drug_inventory': ['last_restock_date'],
    'patient_diagnostic': ['test_date'],
    'billing': ['bill_date'],
}
for table, cols in date_cols_by_table.items():
    for c in cols:
        cleaned_dataframes[table][c] = pd.to_datetime(cleaned_dataframes[table][c])

admission           = cleaned_dataframes['admission']
patient             = cleaned_dataframes['patient']
department          = cleaned_dataframes['department']
ward                = cleaned_dataframes['ward']
bed                 = cleaned_dataframes['bed']
disease             = cleaned_dataframes['disease']
billing             = cleaned_dataframes['billing']
patient_insurance   = cleaned_dataframes['patient_insurance']
insurance_provider  = cleaned_dataframes['insurance_provider']
staff_assignment    = cleaned_dataframes['staff_assignment']
employee            = cleaned_dataframes['employee']

print("HMIS only has 1 hospital modeled (no hospital_id column anywhere).")
print("We use hospital_id=1 / hospital_name='HMIS Hospital' as a placeholder")
print("until a multi-hospital source (e.g. Beds Management dataset) is merged.\n")

HMIS only has 1 hospital modeled (no hospital_id column anywhere).
We use hospital_id=1 / hospital_name='HMIS Hospital' as a placeholder
until a multi-hospital source (e.g. Beds Management dataset) is merged.



In [33]:
# =========================================================
# SCHEMA STATUS TRACKER
# For every required field in the 4 target tables, we mark:
#   AVAILABLE  -> exists directly in an HMIS table
#   DERIVED    -> computed/joined from HMIS tables
#   MISSING    -> HMIS has no source for this at all
# This becomes a printed report AND drives which columns
# actually get filled vs left as NaN in the output CSVs.
# =========================================================

def print_schema_status(table_name, schema_status):
    """schema_status: dict of field -> (status, note)"""
    df = pd.DataFrame([
        {"field": f, "status": s, "note": n}
        for f, (s, n) in schema_status.items()
    ])
    print(f"\n=== Schema status: {table_name} ===")
    display(df)
    print(f"AVAILABLE: {(df.status=='AVAILABLE').sum()} | "
          f"DERIVED: {(df.status=='DERIVED').sum()} | "
          f"MISSING: {(df.status=='MISSING').sum()}")

In [34]:
# =========================================================
# TABLE 1: HOSPITAL OVERVIEW  (grain = 1 row per admission)
# =========================================================

hospital_overview_schema = {
    "admission_id":               ("AVAILABLE", "admission.admission_id"),
    "patient_id":                 ("AVAILABLE", "admission.patient_id"),
    "hospital_id":                ("DERIVED",   "hardcoded 1 — HMIS models a single hospital"),
    "hospital_name":              ("DERIVED",   "hardcoded 'HMIS Hospital' — same reason"),
    "department_id":              ("AVAILABLE", "admission.department_id"),
    "department_name":            ("DERIVED",   "joined from department table"),
    "admission_date":             ("AVAILABLE", "admission.admission_date"),
    "discharge_date":             ("AVAILABLE", "admission.discharge_date"),
    "admission_type":             ("AVAILABLE", "admission.admission_type"),
    "admission_source":           ("MISSING",   "no such column anywhere in HMIS"),
    "bed_id":                     ("AVAILABLE", "admission.bed_id"),
    "bed_type":                   ("DERIVED",   "proxy = ward.ward_type via bed->ward join"),
    "patient_age":                ("DERIVED",   "admission_date - patient.date_of_birth"),
    "patient_gender":              ("AVAILABLE", "patient.gender"),
    "diagnosis":                  ("DERIVED",   "joined from disease.disease_name"),
    "insurance_type":             ("DERIVED",   "patient_insurance policy active on admission_date; only ~24% of admissions have a match, rest = NaN"),
    "total_bill_amount":          ("DERIVED",   "joined from billing.total_amount"),
    "payment_status":             ("DERIVED",   "joined from billing.payment_status"),
    "discharge_status":           ("AVAILABLE", "admission.admission_status — NOTE: only ever 'Discharged' in this dataset, no other outcome recorded"),
    "patient_satisfaction_score": ("MISSING",   "no survey/feedback table in HMIS"),
    "mortality_flag":             ("MISSING",   "no death/outcome field anywhere — admission_status has only 1 unique value"),
    "readmission_flag":           ("DERIVED",   "proxy: same patient re-admitted within 30 days of a prior discharge (not the dedicated Readmission dataset)"),
}
print_schema_status("hospital_overview_dataset", hospital_overview_schema)

# --- build it ---
ho = admission.copy()

# department name
ho = ho.merge(department[['department_id','department_name']], on='department_id', how='left')

# bed_type via bed -> ward -> ward_type
bed_ward = bed.merge(ward[['ward_id','ward_type']], on='ward_id', how='left')
ho = ho.merge(bed_ward[['bed_id','ward_type']].rename(columns={'ward_type':'bed_type'}), on='bed_id', how='left')

# patient_age at time of admission, patient_gender
ho = ho.merge(patient[['patient_id','date_of_birth','gender']], on='patient_id', how='left')
ho['patient_age'] = ((ho['admission_date'] - ho['date_of_birth']).dt.days // 365.25).astype(int)
ho = ho.rename(columns={'gender':'patient_gender'})

# diagnosis
ho = ho.merge(disease[['disease_id','disease_name']], on='disease_id', how='left')
ho = ho.rename(columns={'disease_name':'diagnosis'})

# billing
ho = ho.merge(billing[['admission_id','total_amount','payment_status']], on='admission_id', how='left')
ho = ho.rename(columns={'total_amount':'total_bill_amount'})

# insurance_type: match a policy that was ACTIVE on the admission date
pi = patient_insurance.merge(insurance_provider[['insurance_provider_id','provider_type']], on='insurance_provider_id')
pi_candidates = ho[['admission_id','patient_id','admission_date']].merge(pi, on='patient_id', how='left')
active_policy = pi_candidates[
    (pi_candidates['admission_date'] >= pi_candidates['policy_start_date']) &
    (pi_candidates['admission_date'] <= pi_candidates['policy_end_date'])
].drop_duplicates(subset='admission_id')  # keep first if overlap
ho = ho.merge(active_policy[['admission_id','provider_type']], on='admission_id', how='left')
ho = ho.rename(columns={'provider_type':'insurance_type'})
match_rate = ho['insurance_type'].notna().mean() * 100
print(f"insurance_type matched for {match_rate:.1f}% of admissions (rest left as NaN)")

# hospital_id / hospital_name placeholders
ho['hospital_id'] = 1
ho['hospital_name'] = 'HMIS Hospital'

# readmission_flag proxy: same patient re-admitted within 30 days of prior discharge
ho = ho.sort_values(['patient_id','admission_date'])
ho['prev_discharge'] = ho.groupby('patient_id')['discharge_date'].shift(1)
gap_days = (ho['admission_date'] - ho['prev_discharge']).dt.days
ho['readmission_flag'] = ((gap_days >= 0) & (gap_days <= 30)).astype(int)
ho = ho.drop(columns=['prev_discharge'])
print(f"readmission_flag: {ho['readmission_flag'].sum()} flagged as readmissions (proxy, not the dedicated dataset)")

# columns that genuinely cannot be built from HMIS — kept as NaN placeholders
ho['admission_source'] = pd.NA
ho['patient_satisfaction_score'] = pd.NA
ho['mortality_flag'] = pd.NA
ho = ho.rename(columns={'admission_status':'discharge_status'})

hospital_overview_dataset = ho[[
    'admission_id','patient_id','hospital_id','hospital_name','department_id','department_name',
    'admission_date','discharge_date','admission_type','admission_source','bed_id','bed_type',
    'patient_age','patient_gender','diagnosis','insurance_type','total_bill_amount','payment_status',
    'discharge_status','patient_satisfaction_score','mortality_flag','readmission_flag'
]]
print(f"\nhospital_overview_dataset shape: {hospital_overview_dataset.shape}")


=== Schema status: hospital_overview_dataset ===


,field,status,note
0,admission_id,AVAILABLE,admission.admission_id
1,patient_id,AVAILABLE,admission.patient_id
2,hospital_id,DERIVED,hardcoded 1 — HMIS models a single hospital
3,hospital_name,DERIVED,hardcoded 'HMIS Hospital' — same reason
4,department_id,AVAILABLE,admission.department_id
5,department_name,DERIVED,joined from department table
6,admission_date,AVAILABLE,admission.admission_date
7,discharge_date,AVAILABLE,admission.discharge_date
8,admission_type,AVAILABLE,admission.admission_type
9,admission_source,MISSING,no such column anywhere in HMIS


AVAILABLE: 9 | DERIVED: 10 | MISSING: 3
insurance_type matched for 24.0% of admissions (rest left as NaN)
readmission_flag: 986 flagged as readmissions (proxy, not the dedicated dataset)

hospital_overview_dataset shape: (45000, 22)


In [35]:
# =========================================================
# TABLE 2: PATIENT FLOW  (grain = 1 row per movement event)
# =========================================================

patient_flow_schema = {
    "movement_id":               ("DERIVED", "generated sequential id"),
    "admission_id":               ("AVAILABLE", "admission.admission_id"),
    "patient_id":                 ("AVAILABLE", "admission.patient_id"),
    "hospital_id":                ("DERIVED", "hardcoded 1, same as Table 1"),
    "movement_sequence":          ("DERIVED", "1=Admission event, 2=Discharge event"),
    "movement_type":              ("DERIVED", "'Admission' or 'Discharge' only"),
    "from_department_id/name":    ("MISSING", "no transfer log — HMIS never records a patient moving between departments"),
    "current_department_id/name": ("AVAILABLE", "admission.department_id (fixed for the whole stay)"),
    "bed_id":                     ("AVAILABLE", "admission.bed_id (fixed for the whole stay)"),
    "movement_datetime/date":     ("AVAILABLE", "admission_date or discharge_date"),
    "duration_in_department_hours": ("DERIVED", "only computable for the single (only) department per admission"),
    "year/month/day_of_week/hour_of_day/shift/is_peak_hour": ("DERIVED", "from movement_datetime"),
}
print_schema_status("patient_flow_dataset", patient_flow_schema)
print("\n*** MAJOR GAP: HMIS has no movement/transfer table at all. ***")
print("*** A real patient path like ED -> Ward -> ICU -> Discharge cannot be reconstructed. ***")
print("*** This table is limited to 2 synthetic events per admission: Admission + Discharge. ***\n")

# --- build it (2 rows per admission: Admission event, Discharge event) ---
adm_event = admission[['admission_id','patient_id','department_id','bed_id','admission_date']].copy()
adm_event['movement_type'] = 'Admission'
adm_event['movement_sequence'] = 1
adm_event = adm_event.rename(columns={'admission_date':'movement_datetime'})

dis_event = admission[['admission_id','patient_id','department_id','bed_id','discharge_date']].copy()
dis_event['movement_type'] = 'Discharge'
dis_event['movement_sequence'] = 2
dis_event = dis_event.rename(columns={'discharge_date':'movement_datetime'})

pf = pd.concat([adm_event, dis_event], ignore_index=True)
pf = pf.sort_values(['admission_id','movement_sequence']).reset_index(drop=True)
pf['movement_id'] = range(1, len(pf) + 1)

# department name (same department for both events, no real "from" department exists)
pf = pf.merge(department[['department_id','department_name']], on='department_id', how='left')
pf = pf.rename(columns={'department_id':'current_department_id', 'department_name':'current_department_name'})
pf['from_department_id'] = pd.NA
pf['from_department_name'] = pd.NA

pf['hospital_id'] = 1
pf['movement_date'] = pf['movement_datetime'].dt.date

# duration_in_department_hours: only meaningful for the Admission row = full LOS in hours
los_hours = (admission['discharge_date'] - admission['admission_date']).dt.total_seconds() / 3600
los_map = dict(zip(admission['admission_id'], los_hours))
pf['duration_in_department_hours'] = pf.apply(
    lambda r: los_map[r['admission_id']] if r['movement_type'] == 'Admission' else 0, axis=1
)

# time-derived fields
pf['year'] = pf['movement_datetime'].dt.year
pf['month'] = pf['movement_datetime'].dt.month
pf['day_of_week'] = pf['movement_datetime'].dt.day_name()
pf['hour_of_day'] = pf['movement_datetime'].dt.hour  # note: source dates have no real time component, will be 0
pf['shift'] = pd.cut(pf['hour_of_day'], bins=[-1,7,15,23], labels=['Night','Morning','Evening'])
pf['is_peak_hour'] = pf['hour_of_day'].between(9, 17)

patient_flow_dataset = pf[[
    'movement_id','admission_id','patient_id','hospital_id','movement_sequence','movement_type',
    'from_department_id','current_department_id','from_department_name','current_department_name',
    'bed_id','movement_datetime','movement_date','duration_in_department_hours',
    'year','month','day_of_week','hour_of_day','shift','is_peak_hour'
]]
print(f"patient_flow_dataset shape: {patient_flow_dataset.shape}")
print("NOTE: hour_of_day/shift/is_peak_hour are not meaningful — admission_date/discharge_date")
print("in HMIS carry no time-of-day component, only a date. Would need timestamped source data.")


=== Schema status: patient_flow_dataset ===


,field,status,note
0,movement_id,DERIVED,generated sequential id
1,admission_id,AVAILABLE,admission.admission_id
2,patient_id,AVAILABLE,admission.patient_id
3,hospital_id,DERIVED,"hardcoded 1, same as Table 1"
4,movement_sequence,DERIVED,"1=Admission event, 2=Discharge event"
5,movement_type,DERIVED,'Admission' or 'Discharge' only
6,from_department_id/name,MISSING,no transfer log — HMIS never records a patient...
7,current_department_id/name,AVAILABLE,admission.department_id (fixed for the whole s...
8,bed_id,AVAILABLE,admission.bed_id (fixed for the whole stay)
9,movement_datetime/date,AVAILABLE,admission_date or discharge_date


AVAILABLE: 5 | DERIVED: 6 | MISSING: 1

*** MAJOR GAP: HMIS has no movement/transfer table at all. ***
*** A real patient path like ED -> Ward -> ICU -> Discharge cannot be reconstructed. ***
*** This table is limited to 2 synthetic events per admission: Admission + Discharge. ***

patient_flow_dataset shape: (90000, 20)
NOTE: hour_of_day/shift/is_peak_hour are not meaningful — admission_date/discharge_date
in HMIS carry no time-of-day component, only a date. Would need timestamped source data.


In [36]:
# =========================================================
# TABLE 3: DEPARTMENT ANALYTICS  (grain = hospital+dept+day)
# scoped to the 6 CLINICAL departments — the 5 Admin/Diagnostic
# departments (Radiology, Pathology, Pharmacy, Billing, HR) have
# no wards/beds and never receive admissions, so a bed-based
# daily table for them would be meaningless zeros.
# =========================================================

dept_analytics_schema = {
    "date":                        ("DERIVED", "generated calendar range"),
    "hospital_id/name":            ("DERIVED", "hardcoded"),
    "department_id/name/type":     ("AVAILABLE", "department table"),
    "total_beds":                  ("DERIVED", "sum(ward.total_beds) per department"),
    "occupied_beds_count":         ("DERIVED", "computed from admission_date/discharge_date overlap per day"),
    "bed_occupancy_rate_pct":      ("DERIVED", "occupied / total_beds"),
    "patients_admitted_count":     ("DERIVED", "count of admissions on that date"),
    "patients_discharged_count":   ("DERIVED", "count of discharges on that date"),
    "readmission_count/rate_pct":  ("DERIVED", "same 30-day proxy as Table 1, aggregated by discharge date"),
    "mortality_count/rate_pct":    ("MISSING", "no mortality outcome field exists in HMIS"),
    "avg_length_of_stay_days":     ("DERIVED", "avg LOS of patients discharged that day"),
    "avg_treatment_time_hours":    ("MISSING", "HMIS has no concept of 'treatment time' distinct from LOS"),
    "transfer_events_count":       ("MISSING", "no movement/transfer table, same gap as Table 2"),
    "nurses_on_duty/doctors_on_duty": ("MISSING", "staff_assignment has ward+shift but NO DATE column — it's a static roster, not a daily schedule"),
    "staff_to_patient_ratio":      ("MISSING", "depends on the field above"),
    "equipment_downtime_hours":    ("MISSING", "no equipment table anywhere in HMIS"),
    "avg_satisfaction_score":      ("MISSING", "no survey data"),
    "department_efficiency_score": ("MISSING", "a composite score needs satisfaction + mortality + more, which are missing"),
}
print_schema_status("department_analytics_dataset", dept_analytics_schema)

CLINICAL_DEPT_IDS = ward['department_id'].unique()  # only depts that actually have wards/beds
clinical_departments = department[department['department_id'].isin(CLINICAL_DEPT_IDS)]

# --- occupied_beds_count per department per day (event-delta + cumulative sum trick) ---
events = pd.concat([
    admission[['department_id','admission_date']].rename(columns={'admission_date':'date'}).assign(delta=1),
    admission.assign(day_after_discharge=admission['discharge_date'] + pd.Timedelta(days=1))
             [['department_id','day_after_discharge']].rename(columns={'day_after_discharge':'date'}).assign(delta=-1)
])
daily_delta = events.groupby(['department_id','date'])['delta'].sum().reset_index()

full_date_range = pd.date_range(admission['admission_date'].min(), admission['discharge_date'].max(), freq='D')
idx = pd.MultiIndex.from_product([CLINICAL_DEPT_IDS, full_date_range], names=['department_id','date'])
da = daily_delta.set_index(['department_id','date']).reindex(idx, fill_value=0).reset_index()
da = da.sort_values(['department_id','date'])
da['occupied_beds_count'] = da.groupby('department_id')['delta'].cumsum()
da = da.drop(columns=['delta'])

# total_beds, department name/type
total_beds_by_dept = ward.groupby('department_id')['total_beds'].sum().reset_index()
da = da.merge(total_beds_by_dept, on='department_id', how='left')
da = da.merge(department[['department_id','department_name','department_type']], on='department_id', how='left')
da['bed_occupancy_rate_pct'] = (da['occupied_beds_count'] / da['total_beds'] * 100).round(2)

# patients_admitted_count / patients_discharged_count per department per day
admitted_counts = admission.groupby(['department_id','admission_date']).size().reset_index(name='patients_admitted_count')
admitted_counts = admitted_counts.rename(columns={'admission_date':'date'})
discharged_counts = admission.groupby(['department_id','discharge_date']).size().reset_index(name='patients_discharged_count')
discharged_counts = discharged_counts.rename(columns={'discharge_date':'date'})
da = da.merge(admitted_counts, on=['department_id','date'], how='left')
da = da.merge(discharged_counts, on=['department_id','date'], how='left')
da[['patients_admitted_count','patients_discharged_count']] = da[['patients_admitted_count','patients_discharged_count']].fillna(0).astype(int)

# avg_length_of_stay_days: LOS of patients discharged on that day, by department
adm_los = admission.copy()
adm_los['los_days'] = (adm_los['discharge_date'] - adm_los['admission_date']).dt.days
los_by_day = adm_los.groupby(['department_id','discharge_date'])['los_days'].mean().reset_index()
los_by_day = los_by_day.rename(columns={'discharge_date':'date','los_days':'avg_length_of_stay_days'})
da = da.merge(los_by_day, on=['department_id','date'], how='left')

# readmission_count / rate, aggregated by department + discharge date, reusing Table 1's flag
readmit_by_day = ho.groupby(['department_id', ho['admission_date']])['readmission_flag'].sum()  # placeholder pattern
readmit_agg = ho.groupby(['department_id','discharge_date'])['readmission_flag'].agg(['sum','count']).reset_index()
readmit_agg = readmit_agg.rename(columns={'discharge_date':'date','sum':'readmission_count'})
readmit_agg['readmission_rate_pct'] = (readmit_agg['readmission_count'] / readmit_agg['count'] * 100).round(2)
da = da.merge(readmit_agg[['department_id','date','readmission_count','readmission_rate_pct']], on=['department_id','date'], how='left')

da['hospital_id'] = 1
da['hospital_name'] = 'HMIS Hospital'

# genuinely missing fields — left as NaN placeholders for when other datasets are merged
for col in ['mortality_count','mortality_rate_pct','avg_treatment_time_hours','transfer_events_count',
            'nurses_on_duty','doctors_on_duty','staff_to_patient_ratio','equipment_downtime_hours',
            'avg_satisfaction_score','department_efficiency_score']:
    da[col] = pd.NA

department_analytics_dataset = da[[
    'date','hospital_id','hospital_name','department_id','department_name','department_type',
    'total_beds','occupied_beds_count','bed_occupancy_rate_pct','patients_admitted_count',
    'patients_discharged_count','readmission_count','readmission_rate_pct','mortality_count',
    'mortality_rate_pct','avg_length_of_stay_days','avg_treatment_time_hours','transfer_events_count',
    'nurses_on_duty','doctors_on_duty','staff_to_patient_ratio','equipment_downtime_hours',
    'avg_satisfaction_score','department_efficiency_score'
]]
print(f"department_analytics_dataset shape: {department_analytics_dataset.shape}")


=== Schema status: department_analytics_dataset ===


,field,status,note
0,date,DERIVED,generated calendar range
1,hospital_id/name,DERIVED,hardcoded
2,department_id/name/type,AVAILABLE,department table
3,total_beds,DERIVED,sum(ward.total_beds) per department
4,occupied_beds_count,DERIVED,computed from admission_date/discharge_date ov...
5,bed_occupancy_rate_pct,DERIVED,occupied / total_beds
6,patients_admitted_count,DERIVED,count of admissions on that date
7,patients_discharged_count,DERIVED,count of discharges on that date
8,readmission_count/rate_pct,DERIVED,"same 30-day proxy as Table 1, aggregated by di..."
9,mortality_count/rate_pct,MISSING,no mortality outcome field exists in HMIS


AVAILABLE: 1 | DERIVED: 9 | MISSING: 8
department_analytics_dataset shape: (13224, 24)


In [37]:
# =========================================================
# TABLE 4: RESOURCE UTILIZATION
# grain = hospital+dept+day+resource_type (Bed/Equipment/Staff)
# HMIS can only build the 'Bed' resource_type rows.
# Equipment rows: skipped entirely — no equipment table in HMIS.
# Clinical Staff rows: skipped — staff_assignment has no date
# column, so a per-day staffing count can't be produced.
# =========================================================

resource_util_schema = {
    "resource_utilization_id": ("DERIVED", "generated sequential id"),
    "date/hospital_id/name/department_id/name": ("DERIVED", "same as Table 3"),
    "resource_type":            ("DERIVED", "only 'Bed' rows generated"),
    "resource_category":        ("MISSING", "would need ward_type-level breakdown, kept generic for now"),
    "total_units_available":    ("DERIVED", "total_beds per department"),
    "units_in_use":             ("DERIVED", "occupied_beds_count per department per day"),
    "units_under_maintenance":  ("MISSING", "no such concept exists for beds in HMIS"),
    "utilization_rate_pct":     ("DERIVED", "units_in_use / total_units_available"),
    "shortage_flag":            ("DERIVED", "utilization_rate_pct > 90% (assumed threshold)"),
    "capacity_hours":           ("DERIVED", "total_units_available * 24 (assumes a bed is 'available' all day)"),
    "utilized_hours":           ("DERIVED", "units_in_use * 24 (approximation, not true occupied-hours)"),
    "idle_hours":               ("DERIVED", "capacity_hours - utilized_hours"),
    "downtime_hours":           ("MISSING", "no maintenance/downtime data for beds in HMIS"),
}
print_schema_status("resource_utilization_dataset (Bed rows only)", resource_util_schema)
print("\n*** Equipment and Clinical Staff resource_type rows are NOT generated. ***")
print("*** Equipment needs the Beds Management dataset (per your doc's own mapping). ***")
print("*** Staff needs a dated schedule — staff_assignment only has ward+shift, no date. ***\n")

ru = da[['department_id','department_name','date','total_beds','occupied_beds_count','bed_occupancy_rate_pct']].copy()
ru = ru.rename(columns={'total_beds':'total_units_available','occupied_beds_count':'units_in_use',
                         'bed_occupancy_rate_pct':'utilization_rate_pct'})
ru['resource_type'] = 'Bed'
ru['resource_category'] = pd.NA
ru['hospital_id'] = 1
ru['hospital_name'] = 'HMIS Hospital'
ru['units_under_maintenance'] = pd.NA
ru['shortage_flag'] = ru['utilization_rate_pct'] > 90
ru['capacity_hours'] = ru['total_units_available'] * 24
ru['utilized_hours'] = ru['units_in_use'] * 24
ru['idle_hours'] = ru['capacity_hours'] - ru['utilized_hours']
ru['downtime_hours'] = pd.NA
ru['resource_utilization_id'] = range(1, len(ru) + 1)

resource_utilization_dataset = ru[[
    'resource_utilization_id','date','hospital_id','hospital_name','department_id','department_name',
    'resource_type','resource_category','total_units_available','units_in_use','units_under_maintenance',
    'utilization_rate_pct','shortage_flag','capacity_hours','utilized_hours','idle_hours','downtime_hours'
]]
print(f"resource_utilization_dataset shape: {resource_utilization_dataset.shape}")


=== Schema status: resource_utilization_dataset (Bed rows only) ===


,field,status,note
0,resource_utilization_id,DERIVED,generated sequential id
1,date/hospital_id/name/department_id/name,DERIVED,same as Table 3
2,resource_type,DERIVED,only 'Bed' rows generated
3,resource_category,MISSING,"would need ward_type-level breakdown, kept gen..."
4,total_units_available,DERIVED,total_beds per department
5,units_in_use,DERIVED,occupied_beds_count per department per day
6,units_under_maintenance,MISSING,no such concept exists for beds in HMIS
7,utilization_rate_pct,DERIVED,units_in_use / total_units_available
8,shortage_flag,DERIVED,utilization_rate_pct > 90% (assumed threshold)
9,capacity_hours,DERIVED,total_units_available * 24 (assumes a bed is '...


AVAILABLE: 0 | DERIVED: 10 | MISSING: 3

*** Equipment and Clinical Staff resource_type rows are NOT generated. ***
*** Equipment needs the Beds Management dataset (per your doc's own mapping). ***
*** Staff needs a dated schedule — staff_assignment only has ward+shift, no date. ***

resource_utilization_dataset shape: (13224, 17)


In [38]:
# =========================================================
# SAVE THE FOUR FINAL TABLES
# =========================================================
hospital_overview_dataset.to_csv(PROCESSED_DIR / "hospital_overview_dataset.csv", index=False)
patient_flow_dataset.to_csv(PROCESSED_DIR / "patient_flow_dataset.csv", index=False)
department_analytics_dataset.to_csv(PROCESSED_DIR / "department_analytics_dataset.csv", index=False)
resource_utilization_dataset.to_csv(PROCESSED_DIR / "resource_utilization_dataset.csv", index=False)

print("Saved all 4 tables to", PROCESSED_DIR)

Saved all 4 tables to C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed


I have completed one dataset HMIS NOW I will try to get more value from the other mentioned dataset


In [31]:
# =========================================================
# LOAD RAW + SET UP CLEANED COPY (your starting code)
# =========================================================
file_names = ["patients.csv", "services_weekly.csv", "staff.csv", "staff_schedule.csv"]

PROJECT_ROOT = Path.cwd().parent
Hospital_Beds_Management = PROJECT_ROOT / "data" / "raw" / "Hospital Beds Management"

dataframes = {}
for file_name in file_names:
    file_path = Hospital_Beds_Management / file_name
    df = pd.read_csv(file_path)
    key_name = file_name[0:-4]
    dataframes[key_name] = df

cleaned_dataframes = {
    table_name: df.copy(deep=True)
    for table_name, df in dataframes.items()
}

# =========================================================
# HELPER FUNCTIONS (same ones used for HMIS — reused as-is)
# =========================================================

def strip_string_columns(df):
    """Remove leading/trailing spaces from every text column."""
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
    return df

def convert_to_datetime(df, date_cols):
    """Convert text date columns to real datetime type."""
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

def report_missing(df, table_name):
    pct = (df.isnull().sum() / len(df) * 100).round(2)
    pct = pct[pct > 0]
    if len(pct) > 0:
        print(f"[{table_name}] missing values:\n{pct}\n")
    else:
        print(f"[{table_name}] no missing values")

def report_duplicates(df, table_name):
    n = df.duplicated().sum()
    print(f"[{table_name}] duplicate rows: {n}")

def check_foreign_key(child_df, child_col, parent_df, parent_col, child_name, parent_name):
    orphans = ~child_df[child_col].isin(parent_df[parent_col])
    print(f"[{child_name}.{child_col} -> {parent_name}.{parent_col}] orphan rows: {orphans.sum()}")
    return orphans


# =========================================================
# --- patients.csv ---
# Note: patient_id format (PAT-xxxxxxxx) confirms these are
# NOT the same patients as HMIS. Keep this table separate,
# never attempt a patient-level join with HMIS.
# =========================================================
df = cleaned_dataframes['patients']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['arrival_date', 'departure_date'])

# sanity check: departure should never be before arrival
bad_dates = df[df['departure_date'] < df['arrival_date']]
print(f"[patients] rows with departure before arrival: {len(bad_dates)}")

# sanity check: age should be a realistic human age
bad_age = df[(df['age'] < 0) | (df['age'] > 120)]
print(f"[patients] rows with implausible age: {len(bad_age)}")

# patient_id should be unique (it's meant to be the primary key)
print(f"[patients] duplicate patient_id count: {df['patient_id'].duplicated().sum()}")

# standardize service names to lowercase-with-underscore (matches existing style:
# 'emergency','surgery','general_medicine','ICU' — ICU is the odd one out, keep as-is)
df['service'] = df['service'].str.strip()

report_missing(df, 'patients')
report_duplicates(df, 'patients')
cleaned_dataframes['patients'] = df


# =========================================================
# --- staff.csv ---
# Master roster: staff_id, staff_name, role, service
# =========================================================
df = cleaned_dataframes['staff']
df = strip_string_columns(df)

print(f"[staff] duplicate staff_id count: {df['staff_id'].duplicated().sum()}")
print(f"[staff] roles found: {sorted(df['role'].unique())}")
print(f"[staff] services found: {sorted(df['service'].unique())}")

report_missing(df, 'staff')
report_duplicates(df, 'staff')
cleaned_dataframes['staff'] = df


# =========================================================
# --- services_weekly.csv ---
# Weekly aggregate per service: capacity, demand, satisfaction, morale
# No real calendar date — just week (1-52) + month (1-12).
# =========================================================
df = cleaned_dataframes['services_weekly']
df = strip_string_columns(df)
df['event'] = df['event'].str.lower()  # standardize casing, e.g. 'Flu' vs 'flu'

# sanity check: can't admit more patients than requested
bad_admit = df[df['patients_admitted'] > df['patients_request']]
print(f"[services_weekly] rows where admitted > requested: {len(bad_admit)}")

# sanity check: admitted + refused should roughly equal requested
mismatch = (df['patients_admitted'] + df['patients_refused'] - df['patients_request']).abs()
print(f"[services_weekly] rows where admitted+refused != requested: {(mismatch > 0).sum()}")

# sanity check: can't admit more patients than beds available that week
bad_capacity = df[df['patients_admitted'] > df['available_beds']]
print(f"[services_weekly] rows where admitted > available_beds: {len(bad_capacity)}")

report_missing(df, 'services_weekly')
report_duplicates(df, 'services_weekly')
cleaned_dataframes['services_weekly'] = df


# =========================================================
# --- staff_schedule.csv ---
# One row per staff per week, with a present (0/1) flag.
# This is the DATED staff schedule that HMIS was missing.
# =========================================================
df = cleaned_dataframes['staff_schedule']
df = strip_string_columns(df)

# present should only ever be 0 or 1
print(f"[staff_schedule] unexpected 'present' values: {sorted(df['present'].unique())}")

# every staff_id here should exist in the staff.csv master list —
# profiling showed 126 unique staff_id here vs 110 in staff.csv,
# so this WILL show orphans. Worth investigating before using it.
check_foreign_key(df, 'staff_id', cleaned_dataframes['staff'], 'staff_id', 'staff_schedule', 'staff')

# each staff member should appear exactly once per week (no dup schedule rows)
dup_schedule = df.duplicated(subset=['staff_id', 'week']).sum()
print(f"[staff_schedule] duplicate (staff_id, week) rows: {dup_schedule}")

report_missing(df, 'staff_schedule')
report_duplicates(df, 'staff_schedule')
cleaned_dataframes['staff_schedule'] = df


# =========================================================
# SAVE — prefixed 'beds_' so these never collide with the
# similarly-named-but-differently-shaped HMIS tables
# (e.g. this 'patients' vs HMIS's 'patient')
# =========================================================
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "Hospital Beds Management"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for table_name, df in cleaned_dataframes.items():
    out_path = PROCESSED_DIR / f"beds_{table_name}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved {table_name} -> {out_path}")

[patients] rows with departure before arrival: 0
[patients] rows with implausible age: 0
[patients] duplicate patient_id count: 0
[patients] no missing values
[patients] duplicate rows: 0
[staff] duplicate staff_id count: 0
[staff] roles found: ['doctor', 'nurse', 'nursing_assistant']
[staff] services found: ['ICU', 'emergency', 'general_medicine', 'surgery']
[staff] no missing values
[staff] duplicate rows: 0
[services_weekly] rows where admitted > requested: 0
[services_weekly] rows where admitted+refused != requested: 0
[services_weekly] rows where admitted > available_beds: 0
[services_weekly] no missing values
[services_weekly] duplicate rows: 0
[staff_schedule] unexpected 'present' values: [np.int64(0), np.int64(1)]
[staff_schedule.staff_id -> staff.staff_id] orphan rows: 6552
[staff_schedule] duplicate (staff_id, week) rows: 0
[staff_schedule] no missing values
[staff_schedule] duplicate rows: 0
Saved patients -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\

In [22]:
# =========================================================
# LOAD CLEANED BEDS MANAGEMENT TABLES
# =========================================================
from pathlib import Path
import pandas as pd
PROJECT_ROOT=Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BEDS_PROCESSED_DIR = (
    PROCESSED_DIR
    / "Hospital Beds Management"
)

beds_patients        = pd.read_csv(BEDS_PROCESSED_DIR / "beds_patients.csv")
beds_services_weekly = pd.read_csv(BEDS_PROCESSED_DIR / "beds_services_weekly.csv")
beds_staff           = pd.read_csv(BEDS_PROCESSED_DIR / "beds_staff.csv")
beds_staff_schedule  = pd.read_csv(BEDS_PROCESSED_DIR / "beds_staff_schedule.csv")

# patients.csv already has real calendar dates -> re-parse them
# (CSV always saves dates as text, so this must be redone on every load)
beds_patients['arrival_date']   = pd.to_datetime(beds_patients['arrival_date'])
beds_patients['departure_date'] = pd.to_datetime(beds_patients['departure_date'])

print("beds_patients:", beds_patients.shape)
print("beds_services_weekly:", beds_services_weekly.shape)
print("beds_staff:", beds_staff.shape)
print("beds_staff_schedule:", beds_staff_schedule.shape)

beds_patients: (1000, 7)
beds_services_weekly: (208, 10)
beds_staff: (110, 4)
beds_staff_schedule: (6552, 6)


In [23]:
# =========================================================
# LOAD THE FOUR FINAL TABLES + THE HMIS department TABLE
# =========================================================
hospital_overview_dataset    = pd.read_csv(PROCESSED_DIR / "hospital_overview_dataset.csv")
patient_flow_dataset         = pd.read_csv(PROCESSED_DIR / "patient_flow_dataset.csv")
department_analytics_dataset = pd.read_csv(PROCESSED_DIR / "department_analytics_dataset.csv")
resource_utilization_dataset = pd.read_csv(PROCESSED_DIR / "resource_utilization_dataset.csv")
department                   = pd.read_csv(PROCESSED_DIR / "Hospital HMIS Dataset for Healthcare Analytics" / "department.csv")

hospital_overview_dataset['admission_date'] = pd.to_datetime(hospital_overview_dataset['admission_date'])
hospital_overview_dataset['discharge_date'] = pd.to_datetime(hospital_overview_dataset['discharge_date'])
patient_flow_dataset['movement_datetime']   = pd.to_datetime(patient_flow_dataset['movement_datetime'])
department_analytics_dataset['date']        = pd.to_datetime(department_analytics_dataset['date'])
resource_utilization_dataset['date']        = pd.to_datetime(resource_utilization_dataset['date'])

# snapshot row counts NOW - we'll check against these after merging
# to prove the merge didn't duplicate or drop any rows
before_rows = {
    'department_analytics': len(department_analytics_dataset),
    'resource_utilization': len(resource_utilization_dataset),
}
print("Row counts before merge:", before_rows)

Row counts before merge: {'department_analytics': 13224, 'resource_utilization': 13224}


In [24]:
# =========================================================
# STEP 2: DEPARTMENT NAME BRIDGE
# Beds Management has 4 services, HMIS has 6 departments.
# Pediatrics and Orthopedics stay UNMAPPED on purpose - they
# get NaN for anything sourced from this dataset, never a
# guessed value.
# =========================================================
SERVICE_TO_DEPARTMENT_NAME = {
    'emergency': 'Emergency',
    'surgery': 'Surgery',
    'ICU': 'ICU',
    'general_medicine': 'Internal Medicine',   # ASSUMPTION - only reasonable match
}

department_lookup = department[['department_id', 'department_name']]

def map_service_to_department_id(service_value):
    dept_name = SERVICE_TO_DEPARTMENT_NAME.get(service_value)
    if dept_name is None:
        return pd.NA
    match = department_lookup.loc[department_lookup['department_name'] == dept_name, 'department_id']
    return match.iloc[0] if len(match) else pd.NA

print("service -> department_name mapping:")
for k, v in SERVICE_TO_DEPARTMENT_NAME.items():
    print(f"  {k} -> {v}")
print("  Pediatrics  -> no match in Beds Management (stays NaN)")
print("  Orthopedics -> no match in Beds Management (stays NaN)")

service -> department_name mapping:
  emergency -> Emergency
  surgery -> Surgery
  ICU -> ICU
  general_medicine -> Internal Medicine
  Pediatrics  -> no match in Beds Management (stays NaN)
  Orthopedics -> no match in Beds Management (stays NaN)


In [25]:
# =========================================================
# STEP 3: WEEK -> CALENDAR DATE BRIDGE
#
# services_weekly/staff_schedule only give week (1-52) + month
# (1-12), no year, no day. Two earlier attempts failed:
#   v1: assumed week 1 = Jan 1, 7-day blocks -> 104/208 rows
#       disagreed with the dataset's own 'month' column.
#   v2: read 'month' directly (fixed that), but still expanded
#       every week into a fixed 7-day window -> weeks inside a
#       month that don't divide evenly by 7 overlapped, which
#       duplicated (department_id, date) rows downstream.
#
# This version: read 'month' directly (ground truth), AND give
# each week a VARIABLE length so the weeks in a month partition
# it exactly - no overlap, no gap. Both checks below must pass
# before continuing.
# =========================================================
import calendar

ASSUMED_YEAR = 2025

week_month_lookup = beds_services_weekly[['week', 'month']].drop_duplicates()

# check 1: every week must map to exactly one month
weeks_with_multiple_months = week_month_lookup.groupby('week')['month'].nunique()
bad_weeks = weeks_with_multiple_months[weeks_with_multiple_months > 1]
print(f"[check] weeks mapped to more than one month: {len(bad_weeks)}")
if len(bad_weeks) > 0:
    print(bad_weeks)

all_weeks = sorted(week_month_lookup['week'].unique())
print(f"[check] weeks covered: {len(all_weeks)} (expect 52), range {min(all_weeks)}-{max(all_weeks)}")

# build week -> (start_date, length_in_days), partitioning each month exactly
week_date_info = {}

for month, group in week_month_lookup.groupby('month'):
    weeks_in_this_month = sorted(group['week'].tolist())
    n_weeks = len(weeks_in_this_month)
    days_in_this_month = calendar.monthrange(ASSUMED_YEAR, month)[1]

    base_len = days_in_this_month // n_weeks
    remainder = days_in_this_month % n_weeks

    cursor = pd.Timestamp(ASSUMED_YEAR, month, 1)
    for i, week_num in enumerate(weeks_in_this_month):
        this_week_len = base_len + (1 if i < remainder else 0)  # spread leftover days across first weeks
        week_date_info[week_num] = (cursor, this_week_len)
        cursor = cursor + pd.Timedelta(days=this_week_len)

def week_to_start_date(week_number):
    return week_date_info[week_number][0]

def week_to_length(week_number):
    return week_date_info[week_number][1]

beds_services_weekly['week_start_date'] = beds_services_weekly['week'].apply(week_to_start_date)
beds_services_weekly['week_length']     = beds_services_weekly['week'].apply(week_to_length)
beds_staff_schedule['week_start_date']  = beds_staff_schedule['week'].apply(week_to_start_date)
beds_staff_schedule['week_length']      = beds_staff_schedule['week'].apply(week_to_length)

# check 2: month alignment (must be 0)
derived_month = beds_services_weekly['week_start_date'].dt.month
mismatch_count = (derived_month != beds_services_weekly['month']).sum()
print(f"[check] month alignment mismatches: {mismatch_count} (must be 0)")

# check 3: no overlapping dates across weeks (must be 0), full year covered
all_ranges = []
for wk, (start, length) in week_date_info.items():
    for offset in range(length):
        all_ranges.append(start + pd.Timedelta(days=offset))
overlap_count = pd.Series(all_ranges).duplicated().sum()
print(f"[check] overlapping dates across weeks: {overlap_count} (must be 0)")
print(f"[check] total days covered: {len(all_ranges)} (expect 365)")

assert mismatch_count == 0, "STOP: month alignment broken, do not continue"
assert overlap_count == 0, "STOP: week ranges overlap, do not continue"
print("\nDate bridge verified. Safe to continue to Step 4.")

[check] weeks mapped to more than one month: 0
[check] weeks covered: 52 (expect 52), range 1-52
[check] month alignment mismatches: 0 (must be 0)
[check] overlapping dates across weeks: 0 (must be 0)
[check] total days covered: 365 (expect 365)

Date bridge verified. Safe to continue to Step 4.


In [26]:
# =========================================================
# STEP 4: EXPAND WEEKLY ROWS INTO DAILY ROWS
# Uses each week's REAL length (from week_date_info), not a
# hardcoded 7 - this is what fixes the duplication bug.
# =========================================================
def expand_week_to_days(df, value_cols):
    rows = []
    for _, row in df.iterrows():
        for offset in range(int(row['week_length'])):
            new_row = row[value_cols].to_dict()
            new_row['date'] = row['week_start_date'] + pd.Timedelta(days=offset)
            rows.append(new_row)
    return pd.DataFrame(rows)

# --- services_weekly -> daily, mapped to HMIS department_id ---
beds_services_weekly['department_id'] = beds_services_weekly['service'].apply(map_service_to_department_id)

services_daily = expand_week_to_days(
    beds_services_weekly,
    value_cols=['department_id', 'available_beds', 'patients_refused', 'patient_satisfaction', 'week_length']
)
services_daily = services_daily.dropna(subset=['department_id'])
services_daily['department_id'] = services_daily['department_id'].astype(int)

# rename NOW, before merging, so column names never collide with
# existing columns and we never need a merge suffix
services_daily = services_daily.rename(columns={
    'patient_satisfaction': 'new_avg_satisfaction_score',
    'available_beds': 'new_external_benchmark_available_beds',
    'patients_refused': 'new_external_benchmark_patients_refused',
})

dup_check = services_daily.duplicated(subset=['department_id', 'date']).sum()
print(f"[services_daily] duplicate (department_id, date) rows: {dup_check} (must be 0)")
print(f"services_daily shape: {services_daily.shape}")

# --- staff_schedule -> weekly counts per department+role -> daily ---
beds_staff_schedule['department_id'] = beds_staff_schedule['service'].apply(map_service_to_department_id)

print("Roles found in staff_schedule:", sorted(beds_staff_schedule['role'].unique()))

weekly_staff_counts = (
    beds_staff_schedule[beds_staff_schedule['present'] == 1]
    .groupby(['week', 'week_start_date', 'week_length', 'department_id', 'role'])
    .size()
    .reset_index(name='count')
)

staff_pivot = weekly_staff_counts.pivot_table(
    index=['week', 'week_start_date', 'week_length', 'department_id'],
    columns='role', values='count', fill_value=0
).reset_index()

rename_map = {}
if 'doctor' in staff_pivot.columns:
    rename_map['doctor'] = 'new_doctors_on_duty'
if 'nurse' in staff_pivot.columns:
    rename_map['nurse'] = 'new_nurses_on_duty'
for role_col in staff_pivot.columns:
    if role_col not in ('week', 'week_start_date', 'week_length', 'department_id') and role_col not in rename_map:
        rename_map[role_col] = f'new_{role_col}_on_duty'
staff_pivot = staff_pivot.rename(columns=rename_map)

staff_value_cols = ['department_id', 'week_length'] + [c for c in staff_pivot.columns if c.startswith('new_')]
staff_daily = expand_week_to_days(staff_pivot, value_cols=staff_value_cols)
staff_daily = staff_daily.dropna(subset=['department_id'])
staff_daily['department_id'] = staff_daily['department_id'].astype(int)

dup_check = staff_daily.duplicated(subset=['department_id', 'date']).sum()
print(f"[staff_daily] duplicate (department_id, date) rows: {dup_check} (must be 0)")
print(f"staff_daily shape: {staff_daily.shape}")

assert services_daily.duplicated(subset=['department_id', 'date']).sum() == 0, "STOP: services_daily has dup keys"
assert staff_daily.duplicated(subset=['department_id', 'date']).sum() == 0, "STOP: staff_daily has dup keys"
print("\nBoth daily tables verified clean. Safe to continue to Step 5.")

[services_daily] duplicate (department_id, date) rows: 0 (must be 0)
services_daily shape: (1460, 6)
Roles found in staff_schedule: ['doctor', 'nurse', 'nursing_assistant']
[staff_daily] duplicate (department_id, date) rows: 0 (must be 0)
staff_daily shape: (984, 6)

Both daily tables verified clean. Safe to continue to Step 5.


In [27]:
# =========================================================
# STEP 5: MERGE INTO department_analytics_dataset
# LEFT JOIN only - every existing HMIS row is kept exactly as-is.
# combine_first() fills existing NaN placeholders, never
# overwrites a value that's already there.
# =========================================================
da = department_analytics_dataset.copy()

da = da.merge(services_daily, on=['department_id', 'date'], how='left')
da = da.merge(staff_daily, on=['department_id', 'date'], how='left')

da['avg_satisfaction_score'] = da['avg_satisfaction_score'].combine_first(da['new_avg_satisfaction_score'])
if 'new_doctors_on_duty' in da.columns:
    da['doctors_on_duty'] = da['doctors_on_duty'].combine_first(da['new_doctors_on_duty'])
if 'new_nurses_on_duty' in da.columns:
    da['nurses_on_duty'] = da['nurses_on_duty'].combine_first(da['new_nurses_on_duty'])

# bed-demand figures kept as clearly separate benchmark columns -
# never blended into total_beds/occupied_beds_count, since those
# describe a DIFFERENT simulated hospital's physical beds
da['external_benchmark_available_beds'] = da['new_external_benchmark_available_beds']
da['external_benchmark_patients_refused'] = da['new_external_benchmark_patients_refused']

# staff_to_patient_ratio, now fillable where both sides matched
mask = da['nurses_on_duty'].notna() & da['doctors_on_duty'].notna() & (da['patients_admitted_count'] > 0)
da.loc[mask, 'staff_to_patient_ratio'] = (
    (da.loc[mask, 'nurses_on_duty'] + da.loc[mask, 'doctors_on_duty']) / da.loc[mask, 'patients_admitted_count']
)

da = da.drop(columns=[c for c in da.columns if c.startswith('new_')])

# --- integrity check: row count must be UNCHANGED after a left join ---
assert len(da) == before_rows['department_analytics'], \
    f"ROW COUNT CHANGED: {before_rows['department_analytics']} -> {len(da)} — stop and investigate"
print(f"department_analytics_dataset row count unchanged: {len(da)} rows ✓")
print(f"avg_satisfaction_score now filled for {da['avg_satisfaction_score'].notna().sum()} / {len(da)} rows")

department_analytics_dataset = da

department_analytics_dataset row count unchanged: 13224 rows ✓
avg_satisfaction_score now filled for 1460 / 13224 rows


In [28]:
# =========================================================
# STEP 6: MERGE INTO resource_utilization_dataset
# Same rule: benchmark columns added, nothing overwritten.
# =========================================================
ru = resource_utilization_dataset.copy()

bench = services_daily[['department_id', 'date',
                         'new_external_benchmark_available_beds',
                         'new_external_benchmark_patients_refused']]
ru = ru.merge(bench, on=['department_id', 'date'], how='left')
ru = ru.rename(columns={
    'new_external_benchmark_available_beds': 'external_benchmark_available_beds',
    'new_external_benchmark_patients_refused': 'external_benchmark_patients_refused',
})

assert len(ru) == before_rows['resource_utilization'], \
    f"ROW COUNT CHANGED: {before_rows['resource_utilization']} -> {len(ru)} — stop and investigate"
print(f"resource_utilization_dataset row count unchanged: {len(ru)} rows ✓")

resource_utilization_dataset = ru

resource_utilization_dataset row count unchanged: 13224 rows ✓


In [29]:
# =========================================================
# STEP 7: hospital_overview / patient_flow - INTENTIONALLY UNTOUCHED
# No row-level bridge exists between HMIS admissions and Beds
# Management's unrelated patients, so nothing merges here.
# =========================================================
print("hospital_overview_dataset: unchanged, not merged (admission-grain, no bridge)")
print("patient_flow_dataset: unchanged, not merged (admission-grain, no bridge)")

hospital_overview_dataset: unchanged, not merged (admission-grain, no bridge)
patient_flow_dataset: unchanged, not merged (admission-grain, no bridge)


In [30]:
# =========================================================
# STEP 8: SAVE UPDATED TABLES
# =========================================================
department_analytics_dataset.to_csv(PROCESSED_DIR / "department_analytics_dataset.csv", index=False)
resource_utilization_dataset.to_csv(PROCESSED_DIR / "resource_utilization_dataset.csv", index=False)
print("Saved updated department_analytics_dataset.csv and resource_utilization_dataset.csv")

Saved updated department_analytics_dataset.csv and resource_utilization_dataset.csv


In [32]:
# =========================================================
# LOAD RAW
# =========================================================
PROJECT_ROOT = Path.cwd().parent
READMISSION_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "Hospital Data for Patient Readmission Prediction"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "Hospital Data for Patient Readmission Prediction"

df = pd.read_csv(READMISSION_RAW_DIR / "Healthcare Data Analysis for readmission.csv")
cleaned_df = df.copy(deep=True)

report_missing(cleaned_df, 'readmission')      # reused helper - confirms what profiling already showed (0 missing)
report_duplicates(cleaned_df, 'readmission')    # reused helper - confirms 0 dup rows


# =========================================================
# STEP 1: STANDARD CLEANING (reusing your existing helpers)
# =========================================================
cleaned_df = strip_string_columns(cleaned_df)

# rename ID columns NOW, before anything else touches them, so they
# can never accidentally be treated as HMIS's patient_id/doctor_id/
# hospital_id later - these are unrelated ID spaces that happen to
# overlap numerically by coincidence
cleaned_df = cleaned_df.rename(columns={
    'patient_id': 'readm_patient_id',
    'doctor_id': 'readm_doctor_id',
    'hospital_id': 'readm_hospital_id',
})

# standardize categorical casing/whitespace already handled by
# strip_string_columns above - just confirm what values exist
for col in ['patient_gender', 'patient_race', 'hospital_ward',
            'department_referral', 'doctor_specialty', 'discharge_status']:
    print(f"{col}: {sorted(cleaned_df[col].unique())}")


# =========================================================
# STEP 2: CONVERT DATES - BUT KEEP THEM FLAGGED AS UNRELIABLE
# format='mixed' handles the DD-MM-YYYY / DD/MM/YYYY inconsistency
# seen in profiling. dayfirst=True since the DD-MM-YYYY rows outnumber
# the ambiguous ones. errors='coerce' turns anything unparseable into
# NaT rather than crashing.
# =========================================================
date_cols = ['Admission_date', 'patient_checkin_date', 'patient_checkout_date']
for col in date_cols:
    cleaned_df[col] = pd.to_datetime(cleaned_df[col], format='mixed', dayfirst=True, errors='coerce')

print(f"unparseable dates after conversion: "
      f"{cleaned_df[date_cols].isna().sum().to_dict()}")


# =========================================================
# STEP 3: FLAG (DON'T FIX) THE UNRELIABLE OPERATIONAL FIELDS
# We do NOT overwrite, drop, or "correct" these - that would mean
# inventing data we don't actually have. We only mark them so
# nothing downstream mistakes them for verified facts.
# =========================================================
checkout_before_checkin = cleaned_df['patient_checkout_date'] < cleaned_df['patient_checkin_date']
beds_impossible = cleaned_df['occupied_beds'] > cleaned_df['hospital_beds_available']

cleaned_df['dates_reliable'] = ~checkout_before_checkin
cleaned_df['bed_counts_reliable'] = ~beds_impossible
cleaned_df['length_of_stay_reliable'] = (
    (cleaned_df['patient_checkout_date'] - cleaned_df['patient_checkin_date']).dt.days
    == cleaned_df['patient_length_of_stay']
)

print(f"rows with unreliable dates: {(~cleaned_df['dates_reliable']).sum()}")
print(f"rows with unreliable bed counts: {(~cleaned_df['bed_counts_reliable']).sum()}")
print(f"rows with unreliable length_of_stay: {(~cleaned_df['length_of_stay_reliable']).sum()}")

# hospital_id/doctor_id/hospital_name: dataset-wide issue, not per-row,
# so one flag covers the whole table rather than a column
IDENTITY_FIELDS_RELIABLE = False  # hospital_id has 6074 unique values vs 1 hospital_name - not a real ID system
print(f"NOTE: readm_hospital_id, readm_doctor_id, time_slot are NOT treated "
      f"as reliable identifiers - documented here, not silently used later")


# =========================================================
# STEP 4: DOCUMENT THE SATISFACTION SCORE SCALE - NO GUESSING
# =========================================================
print(f"patient_sat_score range: {cleaned_df['patient_sat_score'].min()}"
      f"-{cleaned_df['patient_sat_score'].max()} - scale/meaning undocumented "
      f"by the source, NOT assumed to be 0-100. Left as raw value.")


# =========================================================
# STEP 5: SAVE
# =========================================================
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(PROCESSED_DIR / "readmission_dataset.csv", index=False)
print(f"Saved -> {PROCESSED_DIR / 'readmission_dataset.csv'}")

[readmission] no missing values
[readmission] duplicate rows: 0
patient_gender: ['Female', 'Male']
patient_race: ['Asian', 'Black', 'Hispanic', 'Other', 'White']
hospital_ward: ['ER', 'ICU', 'Maternity', 'Pediatrics', 'Surgery']
department_referral: ['Allergy and Immunology', 'Cardiology', 'Cardiology or Internal Medicine', 'Dermatology', 'Endocrinology', 'Gastroenterology', 'Hepatology', 'Hepatology or Gastroenterology', 'Infectious Disease', 'Infectious Disease or Internal Medicine', 'Nephrology', 'Neurology', 'Oncology', 'Psychiatry', 'Pulmonology', 'Pulmonology or Allergy and Immunology', 'Pulmonology or Infectious Disease', 'Rheumatology', 'Urology or Nephrology']
doctor_specialty: ['Allergy and Immunology', 'Cardiology', 'Cardiology or Internal Medicine', 'Dermatology', 'Endocrinology', 'Gastroenterology', 'Hepatology', 'Hepatology or Gastroenterology', 'Infectious Disease', 'Infectious Disease or Internal Medicine', 'Nephrology', 'Neurology', 'Oncology', 'Psychiatry', 'Pulmonolo

In [39]:
# =========================================================
# LOAD DATA (READ-ONLY — readmission_dataset.csv is never written to)
# =========================================================
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

READMISSION_PROCESSED_PATH = PROCESSED_DIR / "Hospital Data for Patient Readmission Prediction" / "readmission_dataset.csv"
HOSPITAL_OVERVIEW_PATH = PROCESSED_DIR / "hospital_overview_dataset.csv"
DISEASE_PATH = PROCESSED_DIR / "Hospital HMIS Dataset for Healthcare Analytics" / "disease.csv"

readmission_df = pd.read_csv(READMISSION_PROCESSED_PATH)
hospital_overview_dataset = pd.read_csv(HOSPITAL_OVERVIEW_PATH)
disease = pd.read_csv(DISEASE_PATH)

hospital_overview_dataset['admission_date'] = pd.to_datetime(hospital_overview_dataset['admission_date'])
hospital_overview_dataset['discharge_date'] = pd.to_datetime(hospital_overview_dataset['discharge_date'])

# snapshot for the integrity checks later
before_row_count = len(hospital_overview_dataset)
readmission_shape_before = readmission_df.shape
print(f"hospital_overview_dataset before: {before_row_count} rows")
print(f"readmission_dataset loaded (read-only): {readmission_shape_before}")


# =========================================================
# STEP 1: NORMALIZE DISEASE NAMES
# Strips a trailing parenthetical abbreviation only, e.g.
# "Chronic Obstructive Pulmonary Disease (COPD)" -> "Chronic
# Obstructive Pulmonary Disease". Doesn't touch anything else -
# this is on a COPY of the column, never overwrites patient_disease.
# =========================================================
import re

def normalize_disease_name(name):
    return re.sub(r'\s*\([^)]*\)\s*$', '', str(name)).strip()

readmission_df['disease_normalized'] = readmission_df['patient_disease'].apply(normalize_disease_name)
disease['disease_normalized'] = disease['disease_name'].apply(normalize_disease_name)

matched_diseases = set(readmission_df['disease_normalized'].unique()) & set(disease['disease_normalized'].unique())
print(f"Diseases matched after normalization: {len(matched_diseases)}")
print(matched_diseases)


# =========================================================
# STEP 2: COMPUTE PER-DISEASE BENCHMARK RATES
# Only computed for the matched diseases - everything else falls
# back to the dataset-wide average, never a guessed disease-specific
# number.
# =========================================================
per_disease_stats = readmission_df.groupby('disease_normalized').agg(
    benchmark_mortality_rate=('discharge_status', lambda s: (s == 'Deceased').mean()),
    benchmark_readmission_rate=('readmission', 'mean'),
    benchmark_satisfaction_score=('patient_sat_score', lambda s: s.mean() / 16),  # /16: documented assumption, 1600-scale looks SAT-style
    benchmark_sample_size=('patient_disease', 'count')
).reset_index()

overall_stats = {
    'benchmark_mortality_rate': (readmission_df['discharge_status'] == 'Deceased').mean(),
    'benchmark_readmission_rate': readmission_df['readmission'].mean(),
    'benchmark_satisfaction_score': readmission_df['patient_sat_score'].mean() / 16,
    'benchmark_sample_size': len(readmission_df),
}
print(f"\nDataset-wide fallback stats: {overall_stats}")


# =========================================================
# STEP 3: BUILD ONE BENCHMARK ROW PER HMIS DISEASE
# Every HMIS disease gets a row - either its own matched stats,
# or the dataset-wide fallback, clearly labeled either way.
# =========================================================
benchmark_rows = []
for _, row in disease.iterrows():
    hmis_disease_name = row['disease_name']
    normalized = row['disease_normalized']

    if normalized in matched_diseases:
        stats = per_disease_stats.loc[per_disease_stats['disease_normalized'] == normalized].iloc[0]
        benchmark_rows.append({
            'disease_name': hmis_disease_name,
            'benchmark_mortality_rate': stats['benchmark_mortality_rate'],
            'benchmark_readmission_rate': stats['benchmark_readmission_rate'],
            'benchmark_satisfaction_score': stats['benchmark_satisfaction_score'],
            'benchmark_sample_size': stats['benchmark_sample_size'],
            'benchmark_match_type': 'disease-specific',
        })
    else:
        benchmark_rows.append({
            'disease_name': hmis_disease_name,
            'benchmark_mortality_rate': overall_stats['benchmark_mortality_rate'],
            'benchmark_readmission_rate': overall_stats['benchmark_readmission_rate'],
            'benchmark_satisfaction_score': overall_stats['benchmark_satisfaction_score'],
            'benchmark_sample_size': overall_stats['benchmark_sample_size'],
            'benchmark_match_type': 'dataset-average fallback',
        })

disease_outcome_benchmarks = pd.DataFrame(benchmark_rows)
print(f"\ndisease_outcome_benchmarks: {len(disease_outcome_benchmarks)} rows "
      f"({(disease_outcome_benchmarks['benchmark_match_type'] == 'disease-specific').sum()} disease-specific, "
      f"{(disease_outcome_benchmarks['benchmark_match_type'] == 'dataset-average fallback').sum()} fallback)")
display(disease_outcome_benchmarks)

disease_outcome_benchmarks.to_csv(PROCESSED_DIR / "disease_outcome_benchmarks.csv", index=False)
print(f"Saved -> {PROCESSED_DIR / 'disease_outcome_benchmarks.csv'}")


# =========================================================
# STEP 4: LEFT JOIN INTO hospital_overview_dataset
# Made idempotent: if this script already ran before, the loaded
# hospital_overview_dataset.csv may already contain benchmark_*
# columns from that earlier run. Drop them first so re-running
# this cell never produces _x/_y suffixed duplicates.
# LEFT JOIN only - every existing HMIS row is kept exactly as-is.
# =========================================================
benchmark_cols = ['benchmark_mortality_rate', 'benchmark_readmission_rate',
                   'benchmark_satisfaction_score', 'benchmark_sample_size',
                   'benchmark_match_type']

existing_benchmark_cols = [c for c in benchmark_cols if c in hospital_overview_dataset.columns]
if existing_benchmark_cols:
    print(f"Found existing benchmark columns from a prior run, dropping before re-merge: {existing_benchmark_cols}")
    hospital_overview_dataset = hospital_overview_dataset.drop(columns=existing_benchmark_cols)

ho = hospital_overview_dataset.merge(
    disease_outcome_benchmarks,
    left_on='diagnosis', right_on='disease_name',
    how='left'
).drop(columns=['disease_name'])

# --- integrity check: row count must be unchanged ---
assert len(ho) == before_row_count, \
    f"ROW COUNT CHANGED: {before_row_count} -> {len(ho)} — stop and investigate"
print(f"\nhospital_overview_dataset row count unchanged: {len(ho)} rows ✓")

# --- confirm the columns actually exist before printing about them ---
assert 'benchmark_mortality_rate' in ho.columns, "benchmark_mortality_rate missing after merge — check join keys"
print(f"benchmark columns filled for {ho['benchmark_mortality_rate'].notna().sum()} / {len(ho)} rows "
      f"(should be ~all, since every diagnosis has at least the fallback)")

hospital_overview_dataset = ho


# =========================================================
# STEP 5: SAVE + CONFIRM readmission_dataset.csv WAS NEVER TOUCHED
# =========================================================
hospital_overview_dataset.to_csv(HOSPITAL_OVERVIEW_PATH, index=False)
print(f"Saved -> {HOSPITAL_OVERVIEW_PATH}")

# reload from disk and compare to the snapshot taken at the very start -
# proves this notebook never wrote to readmission_dataset.csv
readmission_shape_after = pd.read_csv(READMISSION_PROCESSED_PATH).shape
print(f"\nreadmission_dataset.csv shape check: before={readmission_shape_before}, "
      f"after={readmission_shape_after} -> "
      f"{'UNCHANGED ✓' if readmission_shape_before == readmission_shape_after else 'CHANGED — investigate!'}")

hospital_overview_dataset before: 45000 rows
readmission_dataset loaded (read-only): (10000, 29)
Diseases matched after normalization: 7
{'Urinary Tract Infection', 'Pneumonia', 'Hypertension', 'Stroke', 'Chronic Kidney Disease', 'COVID-19', 'Chronic Obstructive Pulmonary Disease'}

Dataset-wide fallback stats: {'benchmark_mortality_rate': np.float64(0.3319), 'benchmark_readmission_rate': np.float64(0.4142), 'benchmark_satisfaction_score': np.float64(52.987625), 'benchmark_sample_size': 10000}

disease_outcome_benchmarks: 20 rows (7 disease-specific, 13 fallback)


,disease_name,benchmark_mortality_rate,benchmark_readmission_rate,benchmark_satisfaction_score,benchmark_sample_size,benchmark_match_type
0,Acute Myocardial Infarction,0.331900,0.414200,52.987625,10000,dataset-average fallback
1,Stroke,0.349854,0.428571,51.995262,343,disease-specific
2,Road Traffic Accident,0.331900,0.414200,52.987625,10000,dataset-average fallback
3,Sepsis,0.331900,0.414200,52.987625,10000,dataset-average fallback
4,Acute Respiratory Distress,0.331900,0.414200,52.987625,10000,dataset-average fallback
5,Diabetes Mellitus,0.331900,0.414200,52.987625,10000,dataset-average fallback
6,Hypertension,0.320413,0.423773,52.903747,387,disease-specific
7,Chronic Kidney Disease,0.338710,0.408602,53.131720,372,disease-specific
8,Chronic Obstructive Pulmonary Disease,0.362205,0.430446,51.797900,381,disease-specific
9,Anemia,0.331900,0.414200,52.987625,10000,dataset-average fallback


Saved -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\disease_outcome_benchmarks.csv
Found existing benchmark columns from a prior run, dropping before re-merge: ['benchmark_mortality_rate', 'benchmark_readmission_rate', 'benchmark_satisfaction_score', 'benchmark_sample_size', 'benchmark_match_type']

hospital_overview_dataset row count unchanged: 45000 rows ✓
benchmark columns filled for 45000 / 45000 rows (should be ~all, since every diagnosis has at least the fallback)
Saved -> C:\Users\sr189\OneDrive\Desktop\Hospital Management Data Analysis\MedTrack_DV\data\processed\hospital_overview_dataset.csv

readmission_dataset.csv shape check: before=(10000, 29), after=(10000, 29) -> UNCHANGED ✓


In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

HMIS_RAW_DIR         = PROJECT_ROOT / "data" / "raw" / "Hospital HMIS Dataset for Healthcare Analytics" / "hospital_synthetic_shalaka" / "hospital_data"
BEDS_RAW_DIR         = PROJECT_ROOT / "data" / "raw" / "Hospital Beds Management"
READMISSION_RAW_DIR  = PROJECT_ROOT / "data" / "raw" / "Hospital Data for Patient Readmission Prediction"

HMIS_PROCESSED_DIR         = PROCESSED_DIR / "Hospital HMIS Dataset for Healthcare Analytics"
BEDS_PROCESSED_DIR         = PROCESSED_DIR / "Hospital Beds Management"
READMISSION_PROCESSED_DIR  = PROCESSED_DIR / "Hospital Data for Patient Readmission Prediction"

for d in [HMIS_PROCESSED_DIR, BEDS_PROCESSED_DIR, READMISSION_PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def strip_string_columns(df):
    str_cols = df.select_dtypes(include='object').columns
    for col in str_cols:
        df[col] = df[col].astype(str).str.strip()
    return df

def convert_to_datetime(df, date_cols):
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

def report_missing(df, table_name):
    pct = (df.isnull().sum() / len(df) * 100).round(2)
    pct = pct[pct > 0]
    print(f"[{table_name}] missing values:\n{pct}\n" if len(pct) > 0 else f"[{table_name}] no missing values")

def report_duplicates(df, table_name):
    print(f"[{table_name}] duplicate rows: {df.duplicated().sum()}")

def check_foreign_key(child_df, child_col, parent_df, parent_col, child_name, parent_name):
    orphans = ~child_df[child_col].isin(parent_df[parent_col])
    print(f"[{child_name}.{child_col} -> {parent_name}.{parent_col}] orphan rows: {orphans.sum()}")
    return orphans


# ===== PART 1: HMIS (19 tables) — verified: 0 orphans, 0 dup PKs, 0 bad dates on every table =====
hmis_file_names = [
    "department.csv", "patient.csv", "employee.csv", "disease.csv",
    "insurance_provider.csv", "drug_manufacturer.csv",
    "doctor.csv", "ward.csv", "drug.csv", "patient_insurance.csv",
    "bed.csv", "drug_inventory.csv",
    "staff_assignment.csv", "admission.csv",
    "diagnostic_test.csv", "patient_diagnostic.csv", "prescription.csv",
    "billing.csv", "billing_detail.csv",
]
dataframes = {f[:-4]: pd.read_csv(HMIS_RAW_DIR / f) for f in hmis_file_names}
cleaned_dataframes = {k: v.copy(deep=True) for k, v in dataframes.items()}

df = cleaned_dataframes['department']
df = strip_string_columns(df)
report_missing(df, 'department'); report_duplicates(df, 'department')
cleaned_dataframes['department'] = df

df = cleaned_dataframes['patient']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['date_of_birth'])
df['contact_number_clean'] = (
    df['contact_number'].str.split('x').str[0]
    .str.replace(r'\D', '', regex=True).str[-10:]
)
report_missing(df, 'patient'); report_duplicates(df, 'patient')
cleaned_dataframes['patient'] = df

df = cleaned_dataframes['employee']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['date_of_joining'])
report_missing(df, 'employee'); report_duplicates(df, 'employee')
cleaned_dataframes['employee'] = df

df = cleaned_dataframes['disease']
df = strip_string_columns(df)
report_missing(df, 'disease'); report_duplicates(df, 'disease')
cleaned_dataframes['disease'] = df

df = cleaned_dataframes['insurance_provider']
df = strip_string_columns(df)
report_missing(df, 'insurance_provider'); report_duplicates(df, 'insurance_provider')
cleaned_dataframes['insurance_provider'] = df

df = cleaned_dataframes['drug_manufacturer']
df = strip_string_columns(df)
report_missing(df, 'drug_manufacturer'); report_duplicates(df, 'drug_manufacturer')
cleaned_dataframes['drug_manufacturer'] = df

df = cleaned_dataframes['doctor']
df = strip_string_columns(df)
check_foreign_key(df, 'employee_id', cleaned_dataframes['employee'], 'employee_id', 'doctor', 'employee')
report_missing(df, 'doctor'); report_duplicates(df, 'doctor')
cleaned_dataframes['doctor'] = df

df = cleaned_dataframes['ward']
df = strip_string_columns(df)
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'ward', 'department')
report_missing(df, 'ward'); report_duplicates(df, 'ward')
cleaned_dataframes['ward'] = df

df = cleaned_dataframes['drug']
df = strip_string_columns(df)
check_foreign_key(df, 'manufacturer_id', cleaned_dataframes['drug_manufacturer'], 'manufacturer_id', 'drug', 'drug_manufacturer')
report_missing(df, 'drug'); report_duplicates(df, 'drug')
cleaned_dataframes['drug'] = df

df = cleaned_dataframes['patient_insurance']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['policy_start_date', 'policy_end_date'])
check_foreign_key(df, 'patient_id', cleaned_dataframes['patient'], 'patient_id', 'patient_insurance', 'patient')
check_foreign_key(df, 'insurance_provider_id', cleaned_dataframes['insurance_provider'], 'insurance_provider_id', 'patient_insurance', 'insurance_provider')
report_missing(df, 'patient_insurance'); report_duplicates(df, 'patient_insurance')
cleaned_dataframes['patient_insurance'] = df

df = cleaned_dataframes['bed']
df = strip_string_columns(df)
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'bed', 'ward')
report_missing(df, 'bed'); report_duplicates(df, 'bed')
cleaned_dataframes['bed'] = df

df = cleaned_dataframes['drug_inventory']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['last_restock_date'])
check_foreign_key(df, 'drug_id', cleaned_dataframes['drug'], 'drug_id', 'drug_inventory', 'drug')
report_missing(df, 'drug_inventory'); report_duplicates(df, 'drug_inventory')
cleaned_dataframes['drug_inventory'] = df

df = cleaned_dataframes['staff_assignment']
df = strip_string_columns(df)
check_foreign_key(df, 'employee_id', cleaned_dataframes['employee'], 'employee_id', 'staff_assignment', 'employee')
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'staff_assignment', 'ward')
report_missing(df, 'staff_assignment'); report_duplicates(df, 'staff_assignment')
cleaned_dataframes['staff_assignment'] = df

df = cleaned_dataframes['admission']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['admission_date', 'discharge_date'])
print(f"[admission] rows with discharge before admission: {len(df[df['discharge_date'] < df['admission_date']])}")
check_foreign_key(df, 'patient_id', cleaned_dataframes['patient'], 'patient_id', 'admission', 'patient')
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'admission', 'department')
check_foreign_key(df, 'ward_id', cleaned_dataframes['ward'], 'ward_id', 'admission', 'ward')
check_foreign_key(df, 'bed_id', cleaned_dataframes['bed'], 'bed_id', 'admission', 'bed')
check_foreign_key(df, 'disease_id', cleaned_dataframes['disease'], 'disease_id', 'admission', 'disease')
report_missing(df, 'admission'); report_duplicates(df, 'admission')
cleaned_dataframes['admission'] = df

df = cleaned_dataframes['diagnostic_test']
df = strip_string_columns(df)
check_foreign_key(df, 'department_id', cleaned_dataframes['department'], 'department_id', 'diagnostic_test', 'department')
report_missing(df, 'diagnostic_test'); report_duplicates(df, 'diagnostic_test')
cleaned_dataframes['diagnostic_test'] = df

df = cleaned_dataframes['patient_diagnostic']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['test_date'])
check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'patient_diagnostic', 'admission')
check_foreign_key(df, 'test_id', cleaned_dataframes['diagnostic_test'], 'test_id', 'patient_diagnostic', 'diagnostic_test')
check_foreign_key(df, 'doctor_id', cleaned_dataframes['doctor'], 'doctor_id', 'patient_diagnostic', 'doctor')
report_missing(df, 'patient_diagnostic'); report_duplicates(df, 'patient_diagnostic')
cleaned_dataframes['patient_diagnostic'] = df

df = cleaned_dataframes['prescription']
df = strip_string_columns(df)
check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'prescription', 'admission')
check_foreign_key(df, 'drug_id', cleaned_dataframes['drug'], 'drug_id', 'prescription', 'drug')
report_missing(df, 'prescription'); report_duplicates(df, 'prescription')
cleaned_dataframes['prescription'] = df

df = cleaned_dataframes['billing']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['bill_date'])
mismatch = (df['insurance_covered_amount'] + df['patient_payable_amount'] - df['total_amount']).abs() > 1
print(f"[billing] rows where covered+payable != total: {mismatch.sum()}")
check_foreign_key(df, 'admission_id', cleaned_dataframes['admission'], 'admission_id', 'billing', 'admission')
report_missing(df, 'billing'); report_duplicates(df, 'billing')
cleaned_dataframes['billing'] = df

df = cleaned_dataframes['billing_detail']
df = strip_string_columns(df)
df['reference_id'] = df['reference_id'].astype('Int64')  # nullable int; NaN is legitimate (Room/Consultation charges have no drug/test reference)
check_foreign_key(df, 'bill_id', cleaned_dataframes['billing'], 'bill_id', 'billing_detail', 'billing')
report_missing(df, 'billing_detail'); report_duplicates(df, 'billing_detail')  # expect ~60% missing on reference_id - normal, not an error
cleaned_dataframes['billing_detail'] = df

for table_name, df in cleaned_dataframes.items():
    df.to_csv(HMIS_PROCESSED_DIR / f"{table_name}.csv", index=False)
    print(f"Saved {table_name} -> {HMIS_PROCESSED_DIR / f'{table_name}.csv'}")


# ===== PART 2: BEDS MANAGEMENT (4 tables) =====
beds_file_names = ["patients.csv", "services_weekly.csv", "staff.csv", "staff_schedule.csv"]
beds_dataframes = {f[:-4]: pd.read_csv(BEDS_RAW_DIR / f) for f in beds_file_names}
beds_cleaned = {k: v.copy(deep=True) for k, v in beds_dataframes.items()}

df = beds_cleaned['patients']
df = strip_string_columns(df)
df = convert_to_datetime(df, ['arrival_date', 'departure_date'])
print(f"[patients] rows with departure before arrival: {(df['departure_date'] < df['arrival_date']).sum()}")
print(f"[patients] rows with implausible age: {((df['age'] < 0) | (df['age'] > 120)).sum()}")
report_missing(df, 'patients'); report_duplicates(df, 'patients')
beds_cleaned['patients'] = df

df = beds_cleaned['staff']
df = strip_string_columns(df)
print(f"[staff] duplicate staff_id count: {df['staff_id'].duplicated().sum()}")
report_missing(df, 'staff'); report_duplicates(df, 'staff')
beds_cleaned['staff'] = df

df = beds_cleaned['services_weekly']
df = strip_string_columns(df)
df['event'] = df['event'].str.lower()
print(f"[services_weekly] rows where admitted > requested: {(df['patients_admitted'] > df['patients_request']).sum()}")
print(f"[services_weekly] rows where admitted > available_beds: {(df['patients_admitted'] > df['available_beds']).sum()}")
report_missing(df, 'services_weekly'); report_duplicates(df, 'services_weekly')
beds_cleaned['services_weekly'] = df

# --- staff_schedule.csv ---
# FIXED: staff_id has 0% overlap with staff.csv (confirmed by direct set check -
# each file assigns its own random ID independently). The real, working link
# between these two files is staff_name: all 110 names in staff.csv appear in
# staff_schedule.csv, which has 16 additional names staff.csv doesn't have.
# Checking staff_id here would silently report 100% orphans and look broken -
# checking staff_name gives the true, useful picture.
df = beds_cleaned['staff_schedule']
df = strip_string_columns(df)
check_foreign_key(df, 'staff_name', beds_cleaned['staff'], 'staff_name', 'staff_schedule', 'staff')
print(f"[staff_schedule] duplicate (staff_id, week) rows: {df.duplicated(subset=['staff_id','week']).sum()}")
report_missing(df, 'staff_schedule'); report_duplicates(df, 'staff_schedule')
beds_cleaned['staff_schedule'] = df

for table_name, df in beds_cleaned.items():
    df.to_csv(BEDS_PROCESSED_DIR / f"beds_{table_name}.csv", index=False)
    print(f"Saved {table_name} -> {BEDS_PROCESSED_DIR / f'beds_{table_name}.csv'}")


# ===== PART 3: READMISSION (1 table) =====
df = pd.read_csv(READMISSION_RAW_DIR / "Healthcare Data Analysis for readmission.csv")
cleaned_df = df.copy(deep=True)
cleaned_df = strip_string_columns(cleaned_df)
cleaned_df = cleaned_df.rename(columns={
    'patient_id': 'readm_patient_id', 'doctor_id': 'readm_doctor_id', 'hospital_id': 'readm_hospital_id',
})
date_cols = ['Admission_date', 'patient_checkin_date', 'patient_checkout_date']
for col in date_cols:
    cleaned_df[col] = pd.to_datetime(cleaned_df[col], format='mixed', dayfirst=True, errors='coerce')

cleaned_df['dates_reliable'] = ~(cleaned_df['patient_checkout_date'] < cleaned_df['patient_checkin_date'])
cleaned_df['bed_counts_reliable'] = ~(cleaned_df['occupied_beds'] > cleaned_df['hospital_beds_available'])
cleaned_df['length_of_stay_reliable'] = (
    (cleaned_df['patient_checkout_date'] - cleaned_df['patient_checkin_date']).dt.days
    == cleaned_df['patient_length_of_stay']
)
print(f"rows with unreliable dates: {(~cleaned_df['dates_reliable']).sum()} / {len(cleaned_df)}")           # 4750 (47.5%)
print(f"rows with unreliable bed counts: {(~cleaned_df['bed_counts_reliable']).sum()} / {len(cleaned_df)}")  # 2777 (27.8%)
print(f"rows with unreliable length_of_stay: {(~cleaned_df['length_of_stay_reliable']).sum()} / {len(cleaned_df)}")  # 9822 (98.2%)

report_missing(cleaned_df, 'readmission'); report_duplicates(cleaned_df, 'readmission')

cleaned_df.to_csv(READMISSION_PROCESSED_DIR / "readmission_dataset.csv", index=False)
print(f"Saved -> {READMISSION_PROCESSED_DIR / 'readmission_dataset.csv'}")